In [15]:
import pandas as pd
from sqlalchemy import create_engine,text
import requests
import json

### Ingesting Table - 1 - Teams 

In [16]:
password = "dbms%40123"
#https://stackoverflow.com/questions/1423804/writing-a-connection-string-when-password-contains-special-characters

host = "127.0.0.1"
engine = create_engine(
    f"mysql+pymysql://root:{password}@{host}:3306/"
    #mysql+pymysql://<username>:<password>@<host>/<dbname>[?<options>]
)

connection = engine.connect()

### Creating DB

In [17]:
query = text("""

CREATE DATABASE IF NOT EXISTS NHL_Analytics;

""")

with engine.begin() as connection:
    result = connection.execute(query)

engine = create_engine(
    f"mysql+pymysql://root:{password}@{host}:3306/NHL_Analytics"
    #mysql+pymysql://<username>:<password>@<host>/<dbname>[?<options>]
)

connection = engine.connect()

### Creating Tables

In [18]:
#T1 - Teams

create_query_1 = text("""
CREATE TABLE IF NOT EXISTS Teams(
Team_ID INT AUTO_INCREMENT PRIMARY KEY,
Team_Abbrev VARCHAR(10) UNIQUE,
Team_Name VARCHAR(100),
Conference_Name VARCHAR(50),
Division_Name VARCHAR(50),
Logo_URL TEXT)
""")

with engine.begin() as connection:
    result = connection.execute(create_query_1)

#T2 - Standings

create_query_2 = text("""
CREATE TABLE IF NOT EXISTS Standings(
Standing_ID INT AUTO_INCREMENT PRIMARY KEY,
Team_ID INT,
Season VARCHAR(20),
Games_Played INT,
Wins INT,
Losses INT,
OT_Losses INT,
Points INT,
Goals_For INT,
Goals_Against INT,
Home_Wins INT,
Away_Wins INT,
Streak_Type VARCHAR(20),
Streak_Count INT,
FOREIGN KEY (Team_ID) REFERENCES Teams(Team_ID),
UNIQUE (Team_ID, Season));
""")

with engine.begin() as connection:
    result = connection.execute(create_query_2)

#T3 - Players

create_query_3 = text("""
CREATE TABLE IF NOT EXISTS Players(
Player_ID BIGINT PRIMARY KEY,
Team_ID INT,
First_Name VARCHAR(100),
Last_Name VARCHAR(100),
Position VARCHAR(10),
Jersey_Number INT,
Birth_Date Date,
Birth_Country VARCHAR(10),
Height_CM REAL,
Weight_KG REAL,
Shoots_Catches VARCHAR(5),
Headshot_Url TEXT,
FOREIGN KEY (Team_ID) REFERENCES Teams(Team_ID));
""")

with engine.begin() as connection:
    result = connection.execute(create_query_3)

create_query_4 = text("""
CREATE TABLE IF NOT EXISTS Games(
Game_ID BIGINT Primary Key,
Season VARCHAR(20),
Game_Type INT,
Game_Date DATE,
Home_Team_ID INT,
Away_Team_ID INT,
Home_Score INT,
Away_Score INT,
Game_State VARCHAR(20),
Venue_Name VARCHAR(150),
FOREIGN KEY (Home_Team_ID) REFERENCES Teams(Team_ID),
FOREIGN KEY (Away_Team_ID) REFERENCES Teams(Team_ID)
) 
""")

with engine.begin() as connection:
    result = connection.execute(create_query_4)

create_query_5 = text("""
CREATE TABLE IF NOT EXISTS Game_Stats(
Stat_ID INT AUTO_INCREMENT Primary Key,
Game_ID BIGINT,
Player_ID BIGINT,
Team_ID INT,
Goals INT,
Assists INT,
Points INT,
Shots_on_goal INT,
Penalty_Min INT,
Toi VARCHAR(10),
Plus_Minus INT,
FOREIGN KEY (Team_ID) REFERENCES Teams(Team_ID),
FOREIGN KEY (Player_ID) REFERENCES Players(Player_ID),
Unique(Game_ID,Player_ID)
) 
""")

with engine.begin() as connection:
    result = connection.execute(create_query_5)

create_query_6 = text("""
CREATE TABLE IF NOT EXISTS Skater_Season_Stats(
Stat_ID INT AUTO_INCREMENT Primary Key,
Player_ID BIGINT,
Season VARCHAR(20),
Team_ID INT,
Games_Played INT,
Goals INT,
Assists INT,
Points INT,
Plus_Minus INT,
Penalty_Min INT,
Shots INT,
Avg_Toi VARCHAR(10),
FOREIGN KEY (Team_ID) REFERENCES Teams(Team_ID),
FOREIGN KEY (Player_ID) REFERENCES Players(Player_ID),
Unique(Player_ID,Team_ID,Season)
) 
""")

with engine.begin() as connection:
    result = connection.execute(create_query_6)

create_query_7 = text("""
CREATE TABLE IF NOT EXISTS Goalie_Season_Stats(
Stat_ID INT AUTO_INCREMENT Primary Key,
Player_ID BIGINT,
Season VARCHAR(20),
Team_ID INT,
Games_Played INT,
Wins INT,
Losses INT,
OT_Losses INT,
Save_pct FLOAT,
Goals_Against_avg FLOAT,
Shutouts INT,
Saves INT,
FOREIGN KEY (Team_ID) REFERENCES Teams(Team_ID),
FOREIGN KEY (Player_ID) REFERENCES Players(Player_ID),
Unique(Player_ID,Team_ID,Season)
) 
""")

with engine.begin() as connection:
    result = connection.execute(create_query_7)

### Ingesting Data From " https://api-web.nhle.com/v1/standings/now" to Teams Table

In [19]:
team_abbrv_list = []
game_id_list = []
player_id_list = []

In [20]:
#https://docs.python-requests.org/en/latest/user/quickstart/
r = requests.get('https://api-web.nhle.com/v1/standings/now')
x=r.json()

In [21]:


all_team_standings = x["standings"]
with engine.connect() as connection:
    for team in all_team_standings:
        team_Abbrev = team.get("teamAbbrev",{}).get("default")
        team_Name = team.get("teamName",{}).get("default")
        conference_Name = team.get("conferenceName")
        division_Name = team.get("divisionName")
        logo_url = team.get("teamLogo")
        team_abbrv_list.append(team_Abbrev)
        query =  text(f"""INSERT IGNORE INTO Teams (Team_Abbrev, Team_Name, Conference_Name, Division_Name, Logo_URL) VALUES (:abbrev,:name,:conference,:div,:logo)""")
        result = connection.execute(
            query,
            {
                'abbrev':team_Abbrev,
                "name":team_Name,
                "conference":conference_Name,
                "div":division_Name,
                "logo":logo_url
            })
        connection.commit() 

### Ingesting Data From "https://api-web.nhle.com/v1/standings/now" to Standings Table

In [22]:
r = requests.get('https://api-web.nhle.com/v1/standings/now')
x=r.json()

In [23]:
all_team_standings = x["standings"]
with engine.connect() as connection:
    for team in all_team_standings:
        season = team.get("seasonId")
        games_played = team.get("gamesPlayed")
        wins = team.get("wins")
        losses = team.get("losses")
        ot_losses = team.get("otLosses")
        points = team.get("points")
        goals_for = team.get("goalFor")
        goals_against = team.get("goalAgainst")
        home_wins = team.get("homeWins")
        away_wins = team.get("roadWins")
        streak_type = team.get("streakCode")
        streak_count = team.get("streakCount")
        tname = team.get("teamName",{}).get("default")
        logo_url = team.get("teamLogo")

        query =  text(f"""INSERT IGNORE INTO Standings (Team_ID,Season, Games_Played, Wins, Losses, OT_Losses,Points,Goals_For,Goals_Against,Home_Wins,Away_Wins,Streak_Type,Streak_Count) VALUES (:teamID,:season,:games,:wins,:losses,:ot_losses,:points,:goals_for,:goals_against,:home_wins,:away_wins,:streak_type,:streak_count)""")

        team_id = connection.execute(text("""SELECT Team_ID FROM Teams where Team_Name = :tname AND Logo_URL = :url"""),{"tname":tname,"url":logo_url}).scalar()
        result = connection.execute(
            query,
            {
                #:seaosn,:games,:wins,:losses,:ot_losses,:points,:goals_for,:goals_against,:home_wins,:away_wins,:streak_type,:streak_count
                "teamID":team_id,
                "season":season,
                "games":games_played,
                "wins":wins,
                "losses":losses,
                "ot_losses":ot_losses,
                "points":points,
                "goals_for":goals_for,
                "goals_against":goals_against,
                "home_wins":home_wins,
                "away_wins":away_wins,
                "streak_type":streak_type,
                "streak_count":streak_count
            })
        connection.commit() 

### Ingesting Data From "https://api-web.nhle.com/v1/roster/{team_abbrev}/current" to Players Table

In [24]:
with engine.connect() as connection:
    for abbv in team_abbrv_list:
        r = requests.get(f'https://api-web.nhle.com/v1/roster/{abbv}/current')
        x = r.json()

        forward_players = x["forwards"]
        defense_players = x["defensemen"]
        goal_players = x["goalies"]

        for player in forward_players:

            player_id = player.get("id")
            first_name = player.get("firstName", {}).get("default")
            last_name = player.get("lastName", {}).get("default")
            position = player.get("positionCode")
            jersey_number = player.get("sweaterNumber")
            birth_date = player.get("birthDate")
            birth_country = player.get("birthCountry")
            height_cm = player.get("heightInCentimeters")
            weight_kg = player.get("weightInKilograms")
            shoots_catches = player.get("shootsCatches")
            headshot_uri = player.get("headshot")

            query =  text(f"""INSERT IGNORE INTO Players (Player_ID,Team_ID,First_Name,Last_Name, Position,Jersey_Number,Birth_Date,Birth_Country,Height_CM,Weight_KG,Shoots_Catches,Headshot_Url) VALUES (:playerid,:teamID,:fname,:lname,:pos_code,:jnum,:dob,:birth_count,:height_cm,:weight_kg,:shoot_catch,:headshot_uri)""")
            team_id = connection.execute(text("""SELECT Team_ID FROM Teams where Team_Abbrev = :tabv"""),{"tabv":abbv}).scalar()
            result = connection.execute(
                query,
                {
                    #:playerid,:teamID,:fname,:lname,:pos_code,:jnum,:dob,:birth_count,:height_cm,:height_kg,:shoot_catch,:headshot_uri
                    "teamID":team_id,
                    "playerid":player_id,
                    "fname":first_name,
                    "lname":last_name,
                    "pos_code":position,
                    "jnum":jersey_number,
                    "dob":birth_date,
                    "birth_count":birth_country,
                    "height_cm":height_cm,
                    "weight_kg":weight_kg,
                    "shoot_catch":shoots_catches,
                    "headshot_uri":headshot_uri
                })
            connection.commit() 

        for player in defense_players:
            player_id = player.get("id")
            first_name = player.get("firstName", {}).get("default")
            last_name = player.get("lastName", {}).get("default")
            position = player.get("positionCode")
            jersey_number = player.get("sweaterNumber")
            birth_date = player.get("birthDate")
            birth_country = player.get("birthCountry")
            height_cm = player.get("heightInCentimeters")
            weight_kg = player.get("weightInKilograms")
            #shoots_catches = player.get("shootsCatches")
            headshot_uri = player.get("headshot")

            query =  text(f"""INSERT IGNORE INTO Players (Player_ID,Team_ID,First_Name,Last_Name, Position,Jersey_Number,Birth_Date,Birth_Country,Height_CM,Weight_KG,Headshot_Url) VALUES (:playerid,:teamID,:fname,:lname,:pos_code,:jnum,:dob,:birth_count,:height_cm,:weight_kg,:headshot_uri)""")
            team_id = connection.execute(text("""SELECT Team_ID FROM Teams where Team_Abbrev = :tabv"""),{"tabv":abbv}).scalar()
            result = connection.execute(
                query,
                {
                    #:playerid,:teamID,:fname,:lname,:pos_code,:jnum,:dob,:birth_count,:height_cm,:height_kg,:shoot_catch,:headshot_uri
                    "teamID":team_id,
                    "playerid":player_id,
                    "fname":first_name,
                    "lname":last_name,
                    "pos_code":position,
                    "jnum":jersey_number,
                    "dob":birth_date,
                    "birth_count":birth_country,
                    "height_cm":height_cm,
                    "weight_kg":weight_kg,
                    #"shoot_catch":shoots_catches,
                    "headshot_uri":headshot_uri
                })
            connection.commit()

        for player in goal_players:
            player_id = player.get("id")
            first_name = player.get("firstName", {}).get("default")
            last_name = player.get("lastName", {}).get("default")
            position = player.get("positionCode")
            jersey_number = player.get("sweaterNumber")
            birth_date = player.get("birthDate")
            birth_country = player.get("birthCountry")
            height_cm = player.get("heightInCentimeters")
            weight_kg = player.get("weightInKilograms")
            shoots_catches = player.get("shootsCatches")
            headshot_uri = player.get("headshot")

            query =  text(f"""INSERT IGNORE INTO Players (Player_ID,Team_ID,First_Name,Last_Name, Position,Jersey_Number,Birth_Date,Birth_Country,Height_CM,Weight_KG,Shoots_Catches,Headshot_Url) VALUES (:playerid,:teamID,:fname,:lname,:pos_code,:jnum,:dob,:birth_count,:height_cm,:weight_kg,:shoot_catch,:headshot_uri)""")
            team_id = connection.execute(text("""SELECT Team_ID FROM Teams where Team_Abbrev = :tabv"""),{"tabv":abbv}).scalar()
            result = connection.execute(
                query,
                {
                    #:playerid,:teamID,:fname,:lname,:pos_code,:jnum,:dob,:birth_count,:height_cm,:height_kg,:shoot_catch,:headshot_uri
                    "teamID":team_id,
                    "playerid":player_id,
                    "fname":first_name,
                    "lname":last_name,
                    "pos_code":position,
                    "jnum":jersey_number,
                    "dob":birth_date,
                    "birth_count":birth_country,
                    "height_cm":height_cm,
                    "weight_kg":weight_kg,
                    "shoot_catch":shoots_catches,
                    "headshot_uri":headshot_uri
                })
            connection.commit() 

### Ingesting Data FROM "https://api-web.nhle.com/v1/club-schedule-season/{team_abbrev}/20252026" to games table

In [25]:
with engine.connect() as connection:
    for abbv in team_abbrv_list:

        r = requests.get(f'https://api-web.nhle.com/v1/club-schedule-season/{abbv}/20252026')
        x = r.json()
        
        all_games = x["games"]

        for game in all_games:
            game_id = game.get("id")
            season = game.get("season")
            game_type = game.get("gameType")
            game_date = game.get("gameDate")
            home_team_id = game.get("homeTeam",{}).get("id")
            away_team_id = game.get("awayTeam",{}).get("id")
            home_score = game.get("homeTeam",{}).get("score")
            away_score = game.get("awayTeam",{}).get("score")
            game_state = game.get("gameState")
            venue_name = game.get("venue",{}).get("default")
            game_id_list.append(game_id)
            query =  text(f"""INSERT IGNORE INTO Games (Game_ID,Season,Game_Type,Game_Date,Home_Team_ID, Away_Team_ID,Home_Score,Away_Score,Game_State,Venue_Name) VALUES (:gID,:season,:gtype,:gDate,:hId,:aId,:home_score,:away_score,:game_State,:venue_name)""")
            #team_id = connection.execute(text("""SELECT Team_ID FROM Teams where Team_Name = :tname AND Logo_URL = :url"""),{"tname":tname,"url":logo_url}).scalar()
            connection.execute(
                query,
                {
                    #:gID,:season,:gtype,:gDate,:hId,:aId,:home_score,:away_score,:game_State,:venue_name
                    "gID":game_id,
                    "season":season,
                    "gtype":game_type,
                    "gDate":game_date,
                    "hId":home_team_id,
                    "aId":away_team_id,
                    "home_score":home_score,
                    "away_score":away_score,
                    "game_State":game_state,
                    "venue_name":venue_name,
                })
            connection.commit() 

### Ingesting data from "https://api-web.nhle.com/v1/gamecenter/{game_id}/boxscore" to game_stats table

### I coded this below cell by not knowing that we were provided JSON file already in project specs. After this cell I referred to that after TRUNCATING the table 

In [26]:


data_path = r"C:\Users\nand4\OneDrive\Desktop\NHL - Analytics\game_stats.json"
with engine.connect() as connection:
    with open(data_path,"r") as file:
        data = json.load(file)
    for x in data:
        player_id = x.get("player_id")
        team_id = x.get("team_id")
        game_id = x.get("game_id")
        goals = x.get("goals")
        assists = x.get("assists")
        points = x.get("points")
        shots_on_goal = x.get("shots_on_goal")
        penalty_min = x.get("penalty_min")
        toi = x.get("toi")
        plus_minus  = x.get("plus_minus")
        query = text("""INSERT IGNORE INTO game_stats (Game_ID,Player_ID,Team_ID,Goals,Assists,Points,Shots_on_goal,Penalty_Min,Toi,Plus_Minus) values (:gid,:pid,:tid,:goals,:assists,:points,:shots,:pim,:toi,:plus_minus)""")
        connection.execute(
            query,
            {
                "gid":game_id,
                "pid":player_id,
                "tid":team_id,
                "goals":goals,
                "assists":assists,
                "points":points,
                "shots":shots_on_goal,
                "pim":penalty_min,
                "toi":toi,
                "plus_minus":plus_minus
            }
        )
        connection.commit()
    

### Ingesting data from skater_season_stats.json file to skater_season_stats table

In [27]:
data_path = r"C:\Users\nand4\OneDrive\Desktop\NHL - Analytics\skater_season_stats.json"
with engine.connect() as connection:

    with open(data_path,"r") as file:
        data = json.load(file)

    for x in data:
        player_id = x.get("player_id")
        team_id = x.get("team_id")
        season = x.get("season")
        games_played = x.get("games_played")
        goals = x.get("goals")
        assists = x.get("assists")
        points = x.get("points")
        penalty_min = x.get("penalty_min")
        avg_toi = x.get("avg_toi")
        plus_minus  = x.get("plus_minus")
        shots = x.get("shots")
        query = text("""
            INSERT IGNORE INTO skater_season_stats
            (
                Player_ID,
                Team_ID,
                Season,
                Games_Played,
                Goals,
                Assists,
                Points,
                Penalty_Min,
                Avg_TOI,
                Plus_Minus,
                Shots
            )
            VALUES
            (
                :pid,
                :tid,
                :season,
                :games_played,
                :goals,
                :assists,
                :points,
                :penalty_min,
                :avg_toi,
                :plus_minus,
                :shots
            )
        """)

        connection.execute(
            query,
            {
                "pid": player_id,
                "tid": team_id,
                "season": season,
                "games_played": games_played,
                "goals": goals,
                "assists": assists,
                "points": points,
                "penalty_min": penalty_min,
                "avg_toi": avg_toi,
                "plus_minus": plus_minus,
                "shots": shots
            }
        )
        connection.commit()

### Ingesting data from goalie_season_stats.json to goalie_season_stats table

In [28]:
data_path = r"C:\Users\nand4\OneDrive\Desktop\NHL - Analytics\goalie_season_stats.json"
with engine.connect() as connection:

    with open(data_path,"r") as file:
        data = json.load(file)

    for x in data:
        player_id = x.get("player_id")
        team_id = x.get("team_id")
        season = x.get("season")
        games_played = x.get("games_played")
        goals = x.get("goals")
        wins = x.get("wins")
        losses = x.get("losses")
        ot_losses = x.get("ot_losses")
        save_pct = x.get("save_pct")
        goals_against_avg = x.get("goals_against_avg")
        shutouts = x.get("shutouts")
        saves = x.get("saves")

        query = text("""
            INSERT IGNORE INTO goalie_season_stats
            (
                Player_ID,
                Team_ID,
                Season,
                Games_Played,
                Wins,
                Losses,
                OT_Losses,
                Save_Pct,
                Goals_Against_Avg,
                Shutouts,
                Saves
            )
            VALUES
            (
                :pid,
                :tid,
                :season,
                :games_played,
                :wins,
                :losses,
                :ot_losses,
                :save_pct,
                :goals_against_avg,
                :shutouts,
                :saves
            )
        """)

        connection.execute(
            query,
            {
                "pid": player_id,
                "tid": team_id,
                "season": season,
                "games_played": games_played,
                "wins": wins,
                "losses": losses,
                "ot_losses": ot_losses,
                "save_pct": save_pct,
                "goals_against_avg": goals_against_avg,
                "shutouts": shutouts,
                "saves": saves
            }
        )

        connection.commit()